In [5]:
"""1.This cell contains the code loading,checking and reshaping the dataset for further processing""" 

import torch 
from torchvision import transforms,datasets
from torch.utils.data import DataLoader


print(torch.__version__)
print(torch.cuda.is_available())

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #Fallback to CPU if GPU is not available
# if(device.type == "cuda"):
#     print("GPU is available")
#     print("Converting Arrays to Tensors . . . .")


# dataset = datasets.ImageFolder(root="ScratchNet/flowers", transform = transform)


# transform = transforms.Compose([transforms.ToTensor()]) #Converting images to tensors


# #Spliting the dataset into training and testing datasets
# train_dataset = int(0.8*len(dataset)) #80% of the dataset will be used for training
# test_dataset = len(dataset) - train_dataset #20% of the dataset will be used for


# train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True) #Creating a dataloader to load the dataset to GPU in batches

# test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True) #Creating a dataloader to load the dataset in batches




AttributeError: module 'torch' has no attribute 'cuda'

In [93]:
"""This cell is used to shuffle the X and Y labels so that the model is 
actually learning and not memorizing the whole dataset

"""
indices = torch.randperm(len(permuted))
X_shuffled = permuted[indices] / 255.0
Y_shuffled = converted_y_label[indices].long().squeeze()


In [94]:
#This cell splits the shuffled data into Training ans Test sets
#80% of the data is used for training and 20% for testing
train_split = int(0.8 * len(X_shuffled))
X_train = X_shuffled[:train_split]
Y_train = Y_shuffled[:train_split]
X_test = X_shuffled[train_split:]
Y_test = Y_shuffled[train_split:] 
print(len(X_train),len(Y_train),len(X_test),len(Y_test))

# Turns (11197, 150, 150, 3) -> (11197, 3, 150, 150)
X_train = X_train.permute(0, 3, 1, 2).float()
X_test = X_test.permute(0, 3, 1, 2).float()
Y_train = Y_train.long()
Y_test = Y_test.long()

8957 8957 2240 2240


In [96]:
from torch import nn
import torch.optim as optim

class FlowerClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        #Convolutional Layer1 forward pass
        self.cnn_layer = nn.Sequential(nn.Conv2d(in_channels=3, out_channels =32,kernel_size=3,padding=1,stride = 1,bias = True) ,
                                       nn.BatchNorm2d(32),
                                       #Activation Layer
                                       nn.ReLU(),

                                       #Pooling Layer
                                       nn.MaxPool2d(kernel_size=3,stride =1,padding=1))





        self.cnn_layer_2 = nn.Sequential(nn.Conv2d(in_channels=32, out_channels =64,kernel_size=3,padding=1,stride = 1,bias = True),
                    nn.BatchNorm2d(64),
                    nn.ReLU(),
                    nn.MaxPool2d(kernel_size=3,stride =1,padding=1))

        self.cnn_layer_3 = nn.Sequential(nn.Conv2d(in_channels=64, out_channels =128,kernel_size=3,padding=1,stride = 1,bias = True),
                    nn.BatchNorm2d(128),
                    nn.ReLU(),
                    nn.MaxPool2d(kernel_size=3,stride =1,padding=1))

    

        #Neural Network forward pass
        self.flatten = nn.Flatten()


        self.layer_1 = nn.Sequential(nn.Linear(in_features = 28*18*18,out_features =128,bias=True),
                                     nn.ReLU())
        self.layer_2 = nn.Sequential(nn.Linear(in_features = 128,out_features =64,bias=True),
                                             nn.ReLU())
        self.layer_3 = nn.Sequential(nn.Linear(in_features = 64,out_features =32,bias=True),
                                             nn.ReLU())

        #Output Layer
        self.out = nn.Linear(in_features = 32, out_features=7)

    

    def forward(self,x):

        x = self.cnn_layer(x)
        x = self.cnn_layer_2(x)
        x = self.cnn_layer_3(x)

        x = (self.layer_1(x))
        x = (self.layer_2(x))
        x = (self.layer_3(x))
        x = self.out(x)
        return x

model = FlowerClassifier().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(model)

FlowerClassifier(
  (cnn_layer): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
  )
  (cnn_layer_2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
  )
  (cnn_layer_3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
  )
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layer_1): 

In [ ]:

epochs = 20

for epoch in range(epochs):

    model.train()
    

    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass & loss
        output = model(images)
        loss = loss_function(output, labels)

        # Optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * images.size(0)
        _, preds = torch.max(output, 1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

    # Step 3: Now both variables exist and can be safely divided
    train_loss = running_train_loss / total_train
    train_acc = (correct_train / total_train) * 100


    model.eval()
    running_test_loss = 0.0
    correct_test = 0
    total_test = 0

    with torch.inference_mode():
        for images, labels in datalaoder:
            images = images.to(device)
            labels = labels.to(device)

            test_output = model(images)
            t_loss = loss_function(test_output, labels)

            running_test_loss += t_loss.item() * images.size(0)
            _, test_preds = torch.max(test_output, 1)
            correct_test += (test_preds == labels).sum().item()
            total_test += labels.size(0)

    test_loss = running_test_loss / total_test
    test_acc = (correct_test / total_test) * 100
    
    print(f"Epoch [{epoch+1:02d}/{epochs:02d}] | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

RuntimeError: Given groups=1, weight of size [32, 3, 3, 3], expected input[64, 150, 3, 150] to have 3 channels, but got 150 channels instead